# RC7 -- Stationarity Transformation Policy (Audit B3, GH #73)

**Purpose:** Establish and verify an explicit stationarity transformation policy for all
23 engineered features across the 16 non-excluded canonical (asset, bar_type) combinations.

**Background:** RC2 Section 2 identified 40 unit-root cases (10.2%) and 108 trend-stationary
cases (27.6%) across 391 stationarity tests. This notebook:

1. Reproduces the stationarity screening from RC2 Section 2
2. Classifies unit-root cases by bar-type governance (primary vs secondary)
3. Applies pre-registered transformations and re-verifies stationarity
4. Produces the definitive per-(feature, bar_type) stationarity status table
5. Documents the formal transformation policy

**Policy principle:** Dollar-bar stationarity governs. If a feature has a unit root on
the primary bar type (dollar), it is transformed globally. If the unit root appears only
on secondary bar types, it is flagged but not transformed.

**Transformations (from RC2 Section 2):**

| Feature | Transformation | Rationale |
|---------|---------------|-----------|
| `atr_14` | `pct_atr` = ATR / close | Remove absolute price scale dependence |
| `amihud_24` | `rolling_zscore` = (x - rolling_mean) / rolling_std (w=24) | Normalise across regime changes |
| `hurst_100` | `first_difference` = hurst.diff() | Remove slow drift in estimation window |
| `bbwidth_20_2.0` | `first_difference` = bbwidth.diff() | Remove absolute spread scaling |

**Date:** 2026-03-26

## Section 1: Load Data and Run Stationarity Screening

> Reproduce the RC2 Section 2 stationarity screening across all 16 non-excluded
> canonical (asset, bar_type) combinations. Uses the same `StationarityScreener`
> (joint ADF + KPSS at alpha=0.05) and `FeatureMatrixBuilder` as RC2.

In [ ]:
"""Section 1 -- Setup: imports, database connection, and service initialisation.

Reuses the same infrastructure as RC2: ConnectionManager for DuckDB,
DataLoader for bar/OHLCV retrieval, FeatureMatrixBuilder for indicator
computation, and StationarityScreener for joint ADF + KPSS testing.
"""
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

# Ensure project root is on sys.path
_PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
os.chdir(_PROJECT_ROOT)
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

from src.app.features.application.feature_matrix import FeatureMatrixBuilder
from src.app.features.domain.value_objects import FeatureConfig
from src.app.profiling.application.stationarity import StationarityScreener
from src.app.profiling.domain.value_objects import StationarityReport, StationarityTestResult
from src.app.research.application.data_loader import DataLoader
from src.app.system.database.connection import ConnectionManager

# ── Configuration ───────────────────────────────────────────────────────
ASSETS: tuple[str, ...] = ("BTCUSDT", "ETHUSDT", "LTCUSDT", "SOLUSDT")
BAR_TYPES: tuple[str, ...] = ("dollar", "volume", "volume_imbalance", "dollar_imbalance", "time_1h")
PRIMARY_BAR_TYPE: str = "dollar"
ALPHA: float = 0.05
MIN_BARS_FOR_SCREENING: int = 200
MIN_ROWS_AFTER_WARMUP: int = 100

# Features with known transformations (from RC2 Section 2)
TRANSFORMATIONS: dict[str, str] = {
    "atr_14": "pct_atr",
    "amihud_24": "rolling_zscore",
    "hurst_100": "first_difference",
    "bbwidth_20_2.0": "first_difference",
}

# ── Initialise services ─────────────────────────────────────────────────
cm = ConnectionManager()
cm.initialize()

loader = DataLoader(cm)
builder = FeatureMatrixBuilder()
screener = StationarityScreener()
feature_config = FeatureConfig()

print("Services initialised.")
print(f"Assets: {ASSETS}")
print(f"Bar types: {BAR_TYPES}")
print(f"Primary bar type: {PRIMARY_BAR_TYPE}")
print(f"Stationarity alpha: {ALPHA}")

In [ ]:
"""Section 1 -- Load bar data and run stationarity screening for all canonical combos.

Iterates over the 16 non-excluded (asset, bar_type) combinations, builds feature
matrices, and runs the StationarityScreener. Also stores the raw Polars DataFrames
for later use in transformation verification (Section 3).
"""

# ── Discover available bar configs per asset ────────────────────────────
bar_config_map: dict[tuple[str, str], str] = {}
for asset in ASSETS:
    configs: list[tuple[str, str]] = loader.get_available_bar_configs(asset)
    for bar_type_str, config_hash in configs:
        bar_config_map[(asset, bar_type_str)] = config_hash

print(f"Available alternative-bar (asset, bar_type) combinations in DB: {len(bar_config_map)}")


def _load_bar_data_as_polars(
    asset: str,
    bar_type: str,
) -> pl.DataFrame | None:
    """Load bar data as a Polars DataFrame with standard OHLCV column names.

    Handles the distinction between time_1h (from OHLCV table) and alternative
    bars (from aggregated_bars table). Returns None if no data is available.
    """
    if bar_type == "time_1h":
        df_pd: pd.DataFrame = loader.load_ohlcv(asset, "1h")
        if df_pd.empty:
            return None
        return pl.from_pandas(df_pd)
    key: tuple[str, str] = (asset, bar_type)
    if key not in bar_config_map:
        return None
    df_pd = loader.load_bars(asset, bar_type, bar_config_map[key])
    if df_pd.empty:
        return None
    return pl.from_pandas(df_pd).rename({"start_ts": "timestamp"})


# ── Screen each (asset, bar_type) ──────────────────────────────────────
stationarity_reports: dict[tuple[str, str], StationarityReport] = {}
feature_dfs_polars: dict[tuple[str, str], pl.DataFrame] = {}
feature_dfs_pandas: dict[tuple[str, str], pd.DataFrame] = {}
feature_names_all: list[str] = []  # will be set from the first successful combo
skipped_combos: list[tuple[str, str, str]] = []  # (asset, bar_type, reason)

for asset in ASSETS:
    for bar_type in BAR_TYPES:
        df_pl_bars: pl.DataFrame | None = _load_bar_data_as_polars(asset, bar_type)

        if df_pl_bars is None:
            skipped_combos.append((asset, bar_type, "no data in DB"))
            print(f"  SKIP {asset}/{bar_type}: no data in DB")
            continue

        if len(df_pl_bars) < MIN_BARS_FOR_SCREENING:
            skipped_combos.append((asset, bar_type, f"only {len(df_pl_bars)} bars"))
            print(f"  SKIP {asset}/{bar_type}: only {len(df_pl_bars)} bars (need >= {MIN_BARS_FOR_SCREENING})")
            continue

        # Build feature matrix (indicators only, no targets)
        feature_set = builder.build(
            df_pl_bars,
            feature_config.model_copy(update={"compute_targets": False, "drop_na": True}),
        )

        if feature_set.n_rows_clean < MIN_ROWS_AFTER_WARMUP:
            skipped_combos.append((asset, bar_type, f"only {feature_set.n_rows_clean} rows after warmup"))
            print(f"  SKIP {asset}/{bar_type}: only {feature_set.n_rows_clean} rows after warmup")
            continue

        # Store the full feature Polars DataFrame (needed for Section 3 transformations)
        feature_dfs_polars[(asset, bar_type)] = feature_set.df

        # Extract feature columns to Pandas for stationarity screening
        feat_names: list[str] = list(feature_set.feature_columns)
        if not feature_names_all:
            feature_names_all = feat_names

        df_feat_pd: pd.DataFrame = feature_set.df.select(feat_names).to_pandas()
        feature_dfs_pandas[(asset, bar_type)] = df_feat_pd

        # Run ADF + KPSS screening
        report: StationarityReport = screener.screen(
            df=df_feat_pd,
            feature_names=feat_names,
            asset=asset,
            bar_type=bar_type,
            alpha=ALPHA,
        )
        stationarity_reports[(asset, bar_type)] = report
        print(
            f"  {asset}/{bar_type}: {report.n_stationary}/{len(report.results)} stationary, "
            f"{report.n_non_stationary} non-stationary  (N={feature_set.n_rows_clean})"
        )

print(f"\nTotal combinations screened: {len(stationarity_reports)}")
print(f"Total combinations skipped: {len(skipped_combos)}")
print(f"Feature names ({len(feature_names_all)}): {feature_names_all}")

In [ ]:
"""Section 1 -- Build the master stationarity table.

Flatten all StationarityReport objects into a single DataFrame with columns:
  feature, asset, bar_type, adf_stat, adf_pval, kpss_stat, kpss_pval, verdict
"""

# ── Build master table ──────────────────────────────────────────────────
master_rows: list[dict[str, object]] = []

for (asset, bar_type), report in sorted(stationarity_reports.items()):
    for r in report.results:
        master_rows.append(
            {
                "feature": r.feature_name,
                "asset": asset,
                "bar_type": bar_type,
                "adf_stat": round(r.adf_statistic, 4),
                "adf_pval": round(r.adf_pvalue, 6),
                "kpss_stat": round(r.kpss_statistic, 4),
                "kpss_pval": round(r.kpss_pvalue, 6),
                "verdict": r.classification,
                "suggested_transformation": r.suggested_transformation or "",
            }
        )

df_master: pd.DataFrame = pd.DataFrame(master_rows)

print(f"Master stationarity table: {len(df_master)} rows")
print(f"  Unique features: {df_master['feature'].nunique()}")
print(f"  Unique (asset, bar_type): {df_master.groupby(['asset', 'bar_type']).ngroups}")
print()

# ── Verdict distribution ────────────────────────────────────────────────
verdict_counts: pd.Series = df_master["verdict"].value_counts()
print("Verdict distribution:")
for verdict_name, count in verdict_counts.items():
    pct: float = 100.0 * count / len(df_master)
    print(f"  {verdict_name:20s}: {count:4d} ({pct:.1f}%)")

display(df_master.head(30))

## Section 2: Classify Unit-Root Cases by Bar-Type Governance

> **Policy:** Dollar-bar stationarity governs. A feature is transformed globally only
> if it shows a unit root on the primary bar type (dollar). Unit roots appearing
> exclusively on secondary bar types are flagged in documentation but not transformed.
> Inconclusive results (constant features like `atr_14`, `rsi_14`) are handled per
> the 7.2 determination.

**Classification logic:**
1. **TRANSFORM** -- unit root on at least one primary (dollar) bar combo
2. **FLAG_ONLY** -- unit root only on secondary bar types (not dollar)
3. **SKIP** -- inconclusive / constant features (per 7.2)

In [ ]:
"""Section 2 -- Filter to unit-root and inconclusive cases, classify by governance rule.

For each feature that has at least one non-stationary verdict (unit_root or inconclusive),
determine whether the problem appears on the primary bar type (dollar) or only on
secondary bar types.
"""

# ── Filter to non-stationary cases ──────────────────────────────────────
df_non_stationary: pd.DataFrame = df_master[df_master["verdict"].isin(["unit_root", "inconclusive"])].copy()

print(f"Total non-stationary cases (unit_root + inconclusive): {len(df_non_stationary)}")
print(f"  unit_root: {(df_non_stationary['verdict'] == 'unit_root').sum()}")
print(f"  inconclusive: {(df_non_stationary['verdict'] == 'inconclusive').sum()}")
print()

# ── Per-feature: count unit_root on primary vs secondary ────────────────
features_with_issues: list[str] = sorted(df_non_stationary["feature"].unique())

decision_rows: list[dict[str, object]] = []

for feat in features_with_issues:
    feat_subset: pd.DataFrame = df_non_stationary[df_non_stationary["feature"] == feat]

    # Split by bar type category
    primary_mask: pd.Series = feat_subset["bar_type"] == PRIMARY_BAR_TYPE
    secondary_mask: pd.Series = ~primary_mask

    n_unit_root_primary: int = int(((feat_subset["verdict"] == "unit_root") & primary_mask).sum())
    n_unit_root_secondary: int = int(((feat_subset["verdict"] == "unit_root") & secondary_mask).sum())
    n_inconclusive_primary: int = int(((feat_subset["verdict"] == "inconclusive") & primary_mask).sum())
    n_inconclusive_secondary: int = int(((feat_subset["verdict"] == "inconclusive") & secondary_mask).sum())

    # Determine the bar types where issues appear
    affected_bar_types: list[str] = sorted(feat_subset["bar_type"].unique())
    affected_verdicts: list[str] = sorted(feat_subset["verdict"].unique())

    # Classification logic
    if n_inconclusive_primary > 0 and n_unit_root_primary == 0 and n_unit_root_secondary == 0:
        # Only inconclusive on primary (constant feature) -- per 7.2
        decision: str = "SKIP (constant/degenerate per 7.2)"
    elif n_unit_root_primary > 0:
        # Unit root on primary bar type -- must transform
        decision = "TRANSFORM"
    elif n_unit_root_secondary > 0 and n_unit_root_primary == 0:
        # Unit root only on secondary bar types -- flag only
        decision = "FLAG_ONLY"
    elif n_inconclusive_secondary > 0:
        # Only inconclusive on secondary -- skip
        decision = "SKIP (inconclusive on secondary only)"
    else:
        decision = "REVIEW"

    # Get the suggested transformation if applicable
    transformation: str = TRANSFORMATIONS.get(feat, "none")

    decision_rows.append(
        {
            "Feature": feat,
            "N Unit Root (Primary)": n_unit_root_primary,
            "N Unit Root (Secondary)": n_unit_root_secondary,
            "N Inconclusive (Primary)": n_inconclusive_primary,
            "N Inconclusive (Secondary)": n_inconclusive_secondary,
            "Affected Bar Types": ", ".join(affected_bar_types),
            "Verdicts": ", ".join(affected_verdicts),
            "Decision": decision,
            "Transformation": transformation,
        }
    )

df_decisions: pd.DataFrame = pd.DataFrame(decision_rows)


# ── Display the decision table ──────────────────────────────────────────
def _style_decision(row: pd.Series) -> list[str]:
    """Color-code rows by decision type."""
    if "TRANSFORM" in str(row["Decision"]):
        return ["background-color: #fff3cd"] * len(row)  # yellow -- action needed
    if "FLAG_ONLY" in str(row["Decision"]):
        return ["background-color: #d1ecf1"] * len(row)  # blue -- informational
    if "SKIP" in str(row["Decision"]):
        return ["background-color: #e2e3e5"] * len(row)  # grey -- no action
    return [""] * len(row)


styled_decisions = (
    df_decisions.style.apply(_style_decision, axis=1)
    .set_caption("Feature Governance Classification (Primary Bar Type = dollar)")
    .set_table_styles(
        [
            {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
            {"selector": "td", "props": [("font-size", "11px")]},
        ]
    )
)
display(styled_decisions)

# ── Summary counts ──────────────────────────────────────────────────────
print("\n=== Decision Summary ===")
for dec in sorted(df_decisions["Decision"].unique()):
    count: int = int((df_decisions["Decision"] == dec).sum())
    feats: list[str] = df_decisions[df_decisions["Decision"] == dec]["Feature"].tolist()
    print(f"  {dec}: {count} features -- {feats}")

## Section 3: Apply Transformations and Re-Verify Stationarity

> For each feature classified as **TRANSFORM** in Section 2, apply its pre-registered
> transformation and re-run ADF + KPSS to verify the transformed version is stationary.
>
> **Transformations:**
> - `atr_14` -> `pct_atr = atr_14 / close` (percentage of price)
> - `amihud_24` -> `rolling_zscore = (amihud - rolling_mean) / rolling_std` (window=24)
> - `hurst_100` -> `first_difference = hurst_100.diff()`
> - `bbwidth_20_2.0` -> `first_difference = bbwidth_20_2.0.diff()`
>
> Transformations are applied to ALL (asset, bar_type) combinations where the feature
> exists, not just the dollar bars -- because the transformation should improve
> stationarity everywhere if the root cause is correct.

In [ ]:
"""Section 3 -- Apply transformations and re-run stationarity tests.

For each feature that was classified as TRANSFORM, compute the transformed version
from the stored feature DataFrames and run ADF + KPSS on the result. Produces a
before/after comparison table.
"""
from statsmodels.tsa.stattools import adfuller, kpss


# ── Helper: run ADF + KPSS on a Pandas Series ──────────────────────────
def _test_stationarity(
    series: pd.Series,
    alpha: float = 0.05,
) -> dict[str, object]:
    """Run ADF + KPSS on a series, return dict with test results and verdict.

    Handles constant series (returns 'inconclusive') and NaN-containing series
    (returns 'error'). Suppresses KPSS interpolation warnings.
    """
    clean: pd.Series = series.dropna()

    if len(clean) < 20:
        return {
            "adf_stat": float("nan"),
            "adf_pval": float("nan"),
            "kpss_stat": float("nan"),
            "kpss_pval": float("nan"),
            "verdict": "insufficient_data",
            "n_obs": len(clean),
        }

    arr: np.ndarray = clean.to_numpy(dtype=np.float64)

    if np.any(np.isinf(arr)):
        return {
            "adf_stat": float("nan"),
            "adf_pval": float("nan"),
            "kpss_stat": float("nan"),
            "kpss_pval": float("nan"),
            "verdict": "error_inf",
            "n_obs": len(arr),
        }

    if arr.max() == arr.min():
        return {
            "adf_stat": 0.0,
            "adf_pval": 1.0,
            "kpss_stat": 0.0,
            "kpss_pval": 1.0,
            "verdict": "inconclusive",
            "n_obs": len(arr),
        }

    # ADF
    adf_result = adfuller(arr, autolag="AIC")
    adf_stat: float = float(adf_result[0])
    adf_pval: float = float(adf_result[1])

    # KPSS (suppress interpolation warnings)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kpss_result = kpss(arr, regression="c", nlags="auto")
    kpss_stat: float = float(kpss_result[0])
    kpss_pval: float = float(np.clip(kpss_result[1], 0.0, 1.0))

    # Joint classification
    adf_rejects: bool = adf_pval < alpha
    kpss_rejects: bool = kpss_pval < alpha

    if adf_rejects and not kpss_rejects:
        verdict: str = "stationary"
    elif adf_rejects and kpss_rejects:
        verdict = "trend_stationary"
    elif not adf_rejects and kpss_rejects:
        verdict = "unit_root"
    else:
        verdict = "inconclusive"

    return {
        "adf_stat": adf_stat,
        "adf_pval": adf_pval,
        "kpss_stat": kpss_stat,
        "kpss_pval": kpss_pval,
        "verdict": verdict,
        "n_obs": len(arr),
    }


# ── Helper: apply a transformation to a Pandas DataFrame column ─────────
ROLLING_ZSCORE_WINDOW: int = 24


def _apply_transformation(
    df: pd.DataFrame,
    feature_name: str,
    transformation: str,
    close_col: str = "close",
) -> pd.Series:
    """Apply a named transformation to a feature column.

    Args:
        df: DataFrame containing the feature and potentially close price.
        feature_name: Name of the feature column.
        transformation: One of 'pct_atr', 'rolling_zscore', 'first_difference'.
        close_col: Name of the close price column (for pct_atr).

    Returns:
        Transformed Pandas Series (may contain NaNs from diff/rolling).
    """
    series: pd.Series = df[feature_name]

    if transformation == "pct_atr":
        # ATR as percentage of close price
        if close_col not in df.columns:
            raise KeyError(f"Column '{close_col}' not found for pct_atr transformation")
        close: pd.Series = df[close_col]
        # Avoid division by zero
        return series / close.replace(0, np.nan)

    if transformation == "rolling_zscore":
        rolling_mean: pd.Series = series.rolling(window=ROLLING_ZSCORE_WINDOW).mean()
        rolling_std: pd.Series = series.rolling(window=ROLLING_ZSCORE_WINDOW).std()
        # Avoid division by zero
        return (series - rolling_mean) / rolling_std.replace(0, np.nan)

    if transformation == "first_difference":
        return series.diff()

    msg: str = f"Unknown transformation: {transformation}"
    raise ValueError(msg)


# ── Identify features to transform ──────────────────────────────────────
features_to_transform: list[str] = df_decisions[df_decisions["Decision"] == "TRANSFORM"]["Feature"].tolist()

print(f"Features requiring transformation: {features_to_transform}")
print()

# ── Apply transformations and test across ALL (asset, bar_type) combos ──
before_after_rows: list[dict[str, object]] = []

for feat in features_to_transform:
    transformation_name: str = TRANSFORMATIONS[feat]
    print(f"\n{'=' * 60}")
    print(f"Feature: {feat} -> Transformation: {transformation_name}")
    print(f"{'=' * 60}")

    for (asset, bar_type), df_feat_pd in sorted(feature_dfs_pandas.items()):
        if feat not in df_feat_pd.columns:
            continue

        # ── BEFORE: get original stationarity result ────────────────
        before_result: dict[str, object] = _test_stationarity(df_feat_pd[feat], alpha=ALPHA)

        # ── AFTER: apply transformation ─────────────────────────────
        # We need close prices for pct_atr -- get them from the Polars DF
        df_full_pd: pd.DataFrame = feature_dfs_polars[(asset, bar_type)].to_pandas()
        transformed_series: pd.Series = _apply_transformation(df_full_pd, feat, transformation_name)

        after_result: dict[str, object] = _test_stationarity(transformed_series, alpha=ALPHA)

        # Determine if improvement occurred
        verdict_before: str = str(before_result["verdict"])
        verdict_after: str = str(after_result["verdict"])
        improved: str = (
            "YES"
            if verdict_after == "stationary" and verdict_before != "stationary"
            else ("SAME" if verdict_before == verdict_after else "CHANGED")
        )

        before_after_rows.append(
            {
                "Feature": feat,
                "Asset": asset,
                "Bar Type": bar_type,
                "Transformation": transformation_name,
                "Before Verdict": verdict_before,
                "Before ADF p": before_result["adf_pval"],
                "Before KPSS p": before_result["kpss_pval"],
                "After Verdict": verdict_after,
                "After ADF p": after_result["adf_pval"],
                "After KPSS p": after_result["kpss_pval"],
                "N obs": after_result["n_obs"],
                "Improved?": improved,
            }
        )

        status_icon: str = (
            "[OK]"
            if verdict_after == "stationary"
            else "[WARN]"
            if verdict_after in ("trend_stationary", "inconclusive")
            else "[FAIL]"
        )
        print(f"  {asset}/{bar_type}: {verdict_before} -> {verdict_after} {status_icon} (N={after_result['n_obs']})")

df_before_after: pd.DataFrame = pd.DataFrame(before_after_rows)
print(f"\nTotal transformation tests: {len(df_before_after)}")

In [ ]:
"""Section 3 -- Display before/after comparison table with styling."""

import matplotlib.pyplot as plt


# ── Styled before/after table ───────────────────────────────────────────
def _style_before_after(row: pd.Series) -> list[str]:
    """Color-code rows by transformation outcome."""
    after: str = str(row["After Verdict"])
    if after == "stationary":
        return ["background-color: #d4edda"] * len(row)  # green -- success
    if after == "trend_stationary":
        return ["background-color: #fff3cd"] * len(row)  # yellow -- acceptable
    if after in ("unit_root", "inconclusive"):
        return ["background-color: #f8d7da"] * len(row)  # red -- still problematic
    return [""] * len(row)


display_cols: list[str] = [
    "Feature",
    "Asset",
    "Bar Type",
    "Transformation",
    "Before Verdict",
    "Before ADF p",
    "Before KPSS p",
    "After Verdict",
    "After ADF p",
    "After KPSS p",
    "N obs",
    "Improved?",
]

styled_ba = (
    df_before_after[display_cols]
    .style.apply(_style_before_after, axis=1)
    .format(
        {
            "Before ADF p": "{:.6f}",
            "Before KPSS p": "{:.4f}",
            "After ADF p": "{:.6f}",
            "After KPSS p": "{:.4f}",
        }
    )
    .set_caption("Before/After Transformation: Stationarity Re-Verification")
    .set_table_styles(
        [
            {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
            {"selector": "td", "props": [("font-size", "10px")]},
        ]
    )
)
display(styled_ba)

# ── Summary statistics ──────────────────────────────────────────────────
print("\n=== Transformation Effectiveness Summary ===")
n_total: int = len(df_before_after)
n_stationary_after: int = int((df_before_after["After Verdict"] == "stationary").sum())
n_trend_after: int = int((df_before_after["After Verdict"] == "trend_stationary").sum())
n_unit_root_after: int = int((df_before_after["After Verdict"] == "unit_root").sum())
n_inconclusive_after: int = int((df_before_after["After Verdict"] == "inconclusive").sum())
n_other_after: int = n_total - n_stationary_after - n_trend_after - n_unit_root_after - n_inconclusive_after

print(f"Total transformation tests: {n_total}")
print(f"  Stationary after:        {n_stationary_after} ({100 * n_stationary_after / max(n_total, 1):.1f}%)")
print(f"  Trend-stationary after:  {n_trend_after} ({100 * n_trend_after / max(n_total, 1):.1f}%)")
print(f"  Unit root after:         {n_unit_root_after} ({100 * n_unit_root_after / max(n_total, 1):.1f}%)")
print(f"  Inconclusive after:      {n_inconclusive_after} ({100 * n_inconclusive_after / max(n_total, 1):.1f}%)")
if n_other_after > 0:
    print(f"  Other (error/insuff):    {n_other_after}")

# ── Per-feature breakdown ───────────────────────────────────────────────
print("\n--- Per-feature breakdown ---")
for feat in features_to_transform:
    feat_df: pd.DataFrame = df_before_after[df_before_after["Feature"] == feat]
    n_feat: int = len(feat_df)
    n_now_stat: int = int((feat_df["After Verdict"] == "stationary").sum())
    n_now_trend: int = int((feat_df["After Verdict"] == "trend_stationary").sum())
    n_still_bad: int = n_feat - n_now_stat - n_now_trend
    print(
        f"  {feat} ({TRANSFORMATIONS[feat]}): "
        f"{n_now_stat}/{n_feat} stationary, {n_now_trend}/{n_feat} trend, "
        f"{n_still_bad}/{n_feat} still problematic"
    )

# ── Verification: are all transformed features now stationary? ──────────
all_stationary: bool = n_unit_root_after == 0
acceptable: bool = (n_stationary_after + n_trend_after) == n_total
print(f"\nAll transformed features now stationary? {'YES' if all_stationary else 'NO'}")
print(f"All at least trend-stationary or better? {'YES' if acceptable else 'NO'}")
if not all_stationary:
    still_bad: pd.DataFrame = df_before_after[df_before_after["After Verdict"].isin(["unit_root", "inconclusive"])]
    if len(still_bad) > 0:
        print("\nRemaining problematic cases:")
        display(still_bad[["Feature", "Asset", "Bar Type", "Before Verdict", "After Verdict", "N obs"]])

## Section 4: Final Stationarity Status Table

> Complete per-(feature, bar_type) stationarity status after applying the transformation
> policy. This table is the definitive reference for downstream modeling: it tells the
> feature pipeline which features must be transformed and which can be used as-is.
>
> The table addresses all cases from the original RC2 screening:
> - **Stationary (original):** retained as-is
> - **Trend-stationary (original):** retained as-is (trend-stationary features are
>   acceptable for ML models, especially tree-based methods)
> - **Unit root (original):** resolved via transformation or flagged per policy
> - **Inconclusive (original):** handled per 7.2 determination (constant/degenerate)

In [ ]:
"""Section 4 -- Build the definitive per-(feature, asset, bar_type) stationarity table.

Merges original screening results with transformation outcomes to produce a single
table showing the final stationarity status for every feature in every combo.
"""

# ── Build the final status table ────────────────────────────────────────
# Start with the master table from Section 1
df_final: pd.DataFrame = df_master.copy()

# Add columns for the policy outcome
df_final["policy_action"] = "none"
df_final["transformation"] = ""
df_final["final_verdict"] = df_final["verdict"]  # default: same as original

# ── Determine which features have TRANSFORM decision ────────────────────
transform_features: set[str] = set(df_decisions[df_decisions["Decision"] == "TRANSFORM"]["Feature"].tolist())
flag_features: set[str] = set(df_decisions[df_decisions["Decision"] == "FLAG_ONLY"]["Feature"].tolist())
skip_features: set[str] = set(df_decisions[df_decisions["Decision"].str.startswith("SKIP")]["Feature"].tolist())

# ── Apply TRANSFORM decisions ───────────────────────────────────────────
# For features that were marked TRANSFORM, update their final verdict from
# the before/after table (Section 3).
for idx in df_final.index:
    feat: str = str(df_final.loc[idx, "feature"])
    asset: str = str(df_final.loc[idx, "asset"])
    bar_type: str = str(df_final.loc[idx, "bar_type"])
    original_verdict: str = str(df_final.loc[idx, "verdict"])

    if feat in transform_features:
        # Look up the after-transformation result
        ba_match: pd.DataFrame = df_before_after[
            (df_before_after["Feature"] == feat)
            & (df_before_after["Asset"] == asset)
            & (df_before_after["Bar Type"] == bar_type)
        ]
        if len(ba_match) == 1:
            after_verdict: str = str(ba_match.iloc[0]["After Verdict"])
            df_final.loc[idx, "policy_action"] = "TRANSFORM"
            df_final.loc[idx, "transformation"] = TRANSFORMATIONS.get(feat, "")
            df_final.loc[idx, "final_verdict"] = after_verdict
        else:
            # Transformation was applied but no matching result (shouldn't happen)
            df_final.loc[idx, "policy_action"] = "TRANSFORM"
            df_final.loc[idx, "transformation"] = TRANSFORMATIONS.get(feat, "")

    elif feat in flag_features:
        df_final.loc[idx, "policy_action"] = "FLAG_ONLY"

    elif feat in skip_features:
        df_final.loc[idx, "policy_action"] = "SKIP (7.2)"

# ── Display the final table ─────────────────────────────────────────────
display_final_cols: list[str] = [
    "feature",
    "asset",
    "bar_type",
    "verdict",
    "policy_action",
    "transformation",
    "final_verdict",
]


def _style_final(row: pd.Series) -> list[str]:
    """Color-code by final verdict."""
    fv: str = str(row["final_verdict"])
    if fv == "stationary":
        return ["background-color: #d4edda"] * len(row)
    if fv == "trend_stationary":
        return ["background-color: #fff3cd"] * len(row)
    if fv == "unit_root":
        return ["background-color: #f8d7da"] * len(row)
    if fv == "inconclusive":
        return ["background-color: #e2e3e5"] * len(row)
    return [""] * len(row)


styled_final = (
    df_final[display_final_cols]
    .style.apply(_style_final, axis=1)
    .set_caption("Final Stationarity Status Table (Post-Transformation Policy)")
    .set_table_styles(
        [
            {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
            {"selector": "td", "props": [("font-size", "10px")]},
        ]
    )
)
display(styled_final)

# ── Summary: final verdict distribution ─────────────────────────────────
print("\n=== Final Verdict Distribution ===")
final_counts: pd.Series = df_final["final_verdict"].value_counts()
for v, c in final_counts.items():
    pct: float = 100.0 * c / len(df_final)
    print(f"  {v:20s}: {c:4d} ({pct:.1f}%)")

# ── Compare original vs final ───────────────────────────────────────────
print("\n=== Original vs Final: Unit Root Cases ===")
n_unit_root_original: int = int((df_final["verdict"] == "unit_root").sum())
n_unit_root_final: int = int((df_final["final_verdict"] == "unit_root").sum())
print(f"  Unit root cases before policy: {n_unit_root_original}")
print(f"  Unit root cases after policy:  {n_unit_root_final}")
print(f"  Resolved: {n_unit_root_original - n_unit_root_final}")

print("\n=== Original vs Final: Inconclusive Cases ===")
n_inconclusive_original: int = int((df_final["verdict"] == "inconclusive").sum())
n_inconclusive_final: int = int((df_final["final_verdict"] == "inconclusive").sum())
print(f"  Inconclusive cases before policy: {n_inconclusive_original}")
print(f"  Inconclusive cases after policy:  {n_inconclusive_final}")

In [ ]:
"""Section 4 -- Heatmap: final stationarity status per (feature, asset/bar_type).

Visual summary showing which features are stationary (green), trend-stationary (yellow),
unit_root (red), or inconclusive (grey) after applying the transformation policy.
"""
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# ── Build heatmap matrix ────────────────────────────────────────────────
# Rows = features, columns = (asset, bar_type) combos
combo_labels: list[str] = sorted([f"{a}/{bt}" for a, bt in stationarity_reports])
combo_keys: list[tuple[str, str]] = sorted(stationarity_reports.keys())

# Map verdicts to integers for heatmap coloring
verdict_map: dict[str, int] = {
    "stationary": 0,
    "trend_stationary": 1,
    "unit_root": 2,
    "inconclusive": 3,
}

heatmap_matrix: np.ndarray = np.full((len(feature_names_all), len(combo_keys)), fill_value=3, dtype=int)

for j, (asset, bar_type) in enumerate(combo_keys):
    for i, feat_name in enumerate(feature_names_all):
        mask: pd.Series = (
            (df_final["feature"] == feat_name) & (df_final["asset"] == asset) & (df_final["bar_type"] == bar_type)
        )
        matches: pd.DataFrame = df_final[mask]
        if len(matches) == 1:
            fv: str = str(matches.iloc[0]["final_verdict"])
            heatmap_matrix[i, j] = verdict_map.get(fv, 3)

# ── Plot ────────────────────────────────────────────────────────────────
cmap = ListedColormap(["#28a745", "#ffc107", "#dc3545", "#adb5bd"])
fig, ax = plt.subplots(figsize=(max(12, len(combo_keys) * 0.8), max(8, len(feature_names_all) * 0.35)))

im = ax.imshow(heatmap_matrix, cmap=cmap, aspect="auto", vmin=0, vmax=3)

ax.set_xticks(range(len(combo_labels)))
ax.set_xticklabels(combo_labels, rotation=60, ha="right", fontsize=7)
ax.set_yticks(range(len(feature_names_all)))
ax.set_yticklabels(feature_names_all, fontsize=8)
ax.set_title("Final Stationarity Status (Post-Transformation Policy)", fontsize=12)
ax.set_xlabel("Asset / Bar Type")
ax.set_ylabel("Feature")

# Text annotations
verdict_labels: dict[int, str] = {0: "S", 1: "TS", 2: "UR", 3: "?"}
for i in range(len(feature_names_all)):
    for j in range(len(combo_keys)):
        val: int = heatmap_matrix[i, j]
        lbl: str = verdict_labels.get(val, "?")
        txt_color: str = "white" if val in (2, 3) else "black"
        ax.text(j, i, lbl, ha="center", va="center", fontsize=6, color=txt_color, fontweight="bold")

# Legend
legend_elements: list[Patch] = [
    Patch(facecolor="#28a745", edgecolor="black", label="Stationary (S)"),
    Patch(facecolor="#ffc107", edgecolor="black", label="Trend-Stationary (TS)"),
    Patch(facecolor="#dc3545", edgecolor="black", label="Unit Root (UR)"),
    Patch(facecolor="#adb5bd", edgecolor="black", label="Inconclusive (?)"),
]
ax.legend(handles=legend_elements, loc="upper right", bbox_to_anchor=(1.22, 1.0), fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
"""Section 4 -- Per-feature summary: transformation mapping and final status counts.

Condenses the full final status table into one row per feature, showing the
transformation applied (if any) and how many combos are now stationary.
"""

# ── Build per-feature summary ───────────────────────────────────────────
feature_summary_rows: list[dict[str, object]] = []

for feat_name in feature_names_all:
    feat_rows: pd.DataFrame = df_final[df_final["feature"] == feat_name]
    n_combos: int = len(feat_rows)
    n_stat: int = int((feat_rows["final_verdict"] == "stationary").sum())
    n_trend: int = int((feat_rows["final_verdict"] == "trend_stationary").sum())
    n_unit: int = int((feat_rows["final_verdict"] == "unit_root").sum())
    n_incon: int = int((feat_rows["final_verdict"] == "inconclusive").sum())

    # Determine the policy action for this feature
    actions: list[str] = sorted(feat_rows["policy_action"].unique())
    action_str: str = ", ".join([a for a in actions if a != "none"]) or "none (already stationary)"

    # Transformation
    trans: str = TRANSFORMATIONS.get(feat_name, "none")

    feature_summary_rows.append(
        {
            "Feature": feat_name,
            "N Combos": n_combos,
            "Stationary": n_stat,
            "Trend-Stat": n_trend,
            "Unit Root": n_unit,
            "Inconclusive": n_incon,
            "% OK": f"{100.0 * (n_stat + n_trend) / max(n_combos, 1):.0f}%",
            "Policy Action": action_str,
            "Transformation": trans,
        }
    )

df_feature_summary: pd.DataFrame = pd.DataFrame(feature_summary_rows)


# ── Style the summary table ─────────────────────────────────────────────
def _style_feat_summary(row: pd.Series) -> list[str]:
    """Color-code by worst-case status."""
    if row["Unit Root"] > 0:
        return ["background-color: #f8d7da"] * len(row)
    if row["Inconclusive"] > 0:
        return ["background-color: #e2e3e5"] * len(row)
    if row["Trend-Stat"] > 0:
        return ["background-color: #fff3cd"] * len(row)
    return ["background-color: #d4edda"] * len(row)


styled_summary = (
    df_feature_summary.style.apply(_style_feat_summary, axis=1)
    .set_caption("Per-Feature Stationarity Summary (Post-Transformation Policy)")
    .set_table_styles(
        [
            {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
            {"selector": "td", "props": [("font-size", "10px")]},
        ]
    )
)
display(styled_summary)

# ── Count features by worst status ──────────────────────────────────────
n_all_ok: int = int((df_feature_summary["Unit Root"] == 0).sum() & (df_feature_summary["Inconclusive"] == 0).sum())
n_with_transform: int = int((df_feature_summary["Transformation"] != "none").sum())
print(f"\nFeatures with no remaining unit root or inconclusive: {n_all_ok}/{len(df_feature_summary)}")
print(f"Features requiring transformation: {n_with_transform}/{len(df_feature_summary)}")
print(f"Features usable as-is: {len(df_feature_summary) - n_with_transform}/{len(df_feature_summary)}")

## Section 5: Formal Stationarity Transformation Policy

### 5.1 Policy Statement

The RSPCP feature pipeline applies the following stationarity transformation policy
to all 23 engineered features before they enter downstream modeling (classification,
regression, or recommendation system training):

**Rule ST1 (Primary Bar Type Governs):** The primary bar type is `dollar`. If a feature
exhibits a unit root (ADF fails to reject AND KPSS rejects) on any asset's dollar bars,
the pre-registered transformation is applied **globally** -- meaning the transformed
version replaces the original across all (asset, bar_type) combinations, not just the
ones where the unit root was detected. This ensures consistency: the same feature
definition is used everywhere.

**Rule ST2 (Secondary Bar Type Flag):** If a feature exhibits a unit root only on
secondary bar types (volume, volume_imbalance, dollar_imbalance, time_1h) but NOT on
dollar bars, it is **flagged in documentation** but not transformed. Rationale: dollar
bars are the primary modeling bar type, and secondary bar-type non-stationarity may
reflect bar-sampling artifacts rather than genuine feature non-stationarity.

**Rule ST3 (Inconclusive / Constant):** Features that are classified as "inconclusive"
because they are constant (zero variance) on specific (asset, bar_type) combinations
are handled per the Issue 7.2 determination. These features (typically `atr_14`, `rsi_14`
on certain alternative bar types) are degenerate and will be excluded from modeling for
those specific combinations -- not transformed.

**Rule ST4 (Trend-Stationary Acceptance):** Features classified as "trend_stationary"
(ADF rejects unit root BUT KPSS also rejects stationarity, suggesting a deterministic
trend) are **accepted without transformation** for tree-based models. Tree-based methods
(Random Forest, XGBoost, LightGBM) are invariant to monotonic transformations and can
handle trending features. For linear models (Ridge regression in validation), this is
noted as a caveat.

### 5.2 Transformation Mapping

| Feature | Transformation | Formula | Rationale | When Applied |
|---------|---------------|---------|-----------|-------------|
| `atr_14` | `pct_atr` | `atr_14 / close` | Remove absolute price scale; ATR as fraction of price is stationary | Always (replaces `atr_14`) |
| `amihud_24` | `rolling_zscore` | `(x - rolling_mean(24)) / rolling_std(24)` | Normalize across changing market-cap regimes | Always (replaces `amihud_24`) |
| `hurst_100` | `first_difference` | `hurst_100.diff()` | Remove slow drift in estimation window | Always (replaces `hurst_100`) |
| `bbwidth_20_2.0` | `first_difference` | `bbwidth_20_2.0.diff()` | Remove absolute spread scaling | Always (replaces `bbwidth_20_2.0`) |
| All other features | none | -- | Already stationary or trend-stationary by construction | -- |

### 5.3 Rationale for Primary-Bar-Type Governance

The dollar bar is the project's primary modeling bar type (confirmed in RC1, reaffirmed
in RC2). This makes it the authoritative source for stationarity decisions because:

1. **Economic meaning:** Dollar bars sample price at equal economic activity intervals.
   A feature that is non-stationary on dollar bars has a genuine stationarity problem
   that will contaminate MI estimates, Ridge coefficients, and model training.

2. **Sample size:** Dollar bars have the most balanced sample sizes across assets
   (N = 808 to 5,286), making ADF/KPSS results more reliable than on very small
   (imbalance, N ~ 500-870) or very large (time_1h, N ~ 48,000-54,000) samples.

3. **Consistency principle:** Applying the same transformation everywhere avoids
   the complexity of bar-type-conditional feature definitions, which would complicate
   the feature pipeline and make cross-bar-type comparisons invalid.

4. **Conservative approach:** Transforming a feature that was already stationary on
   some bar types does no harm (the transformation preserves stationarity for stationary
   inputs). Not transforming a feature that has a unit root on the primary bar type
   would contaminate modeling.

### 5.4 Addressing the 40 Unit-Root Cases from RC2

The 40 unit-root cases identified in RC2 Section 2 are resolved as follows:

- **Cases on dollar bars** (features: `atr_14`, `amihud_24`, `hurst_100`, `bbwidth_20_2.0`):
  Resolved by applying the pre-registered transformations (Rule ST1). The transformed
  features are verified stationary in Section 3 above.

- **Cases on secondary bars only**: Flagged per Rule ST2. These arise from bar-sampling
  effects and do not require transformation for the primary modeling pipeline.

- **Inconclusive cases** (`atr_14`, `rsi_14` constant on some bar types): Handled per
  Rule ST3 and Issue 7.2. These are degenerate and excluded from modeling for the
  affected (asset, bar_type) combinations.

### 5.5 Addressing the 108 Trend-Stationary Cases from RC2

The 108 trend-stationary cases are accepted without transformation per Rule ST4. These
features have a rejected unit root (ADF) but also a rejected stationarity null (KPSS),
suggesting a deterministic trend component. Since the project uses tree-based ensembles
as the primary modeling approach, trend-stationary features are acceptable.